In [1]:
from actor_critic import ActorCriticLSTM, ActorCritic
from itertools import count
from collections import namedtuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
from tinyphysics import TinyPhysicsModel, TinyPhysicsSimulator, State, CONTEXT_LENGTH, STEER_RANGE, CONTROL_START_IDX, DEL_T, LAT_ACCEL_COST_MULTIPLIER, MAX_ACC_DELTA
from controllers import CONTROLLERS, PIDController
from environment import Environment

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
SavedAction = namedtuple('SavedAction', ['log_prob', 'value'])

In [4]:
def finish_episode():
    # Calculating losses and performing backprop
    R = 0
    saved_actions = model.saved_actions
    policy_losses = []
    value_losses = []
    returns = []

    for r in model.rewards[::-1]:
        R = r + 0.99 * R # Gamma is 0.99
        returns.insert(0, R)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + eps)

    for (log_prob, value), R in zip(saved_actions, returns):
        advantage = R - value.item()

        policy_losses.append(-log_prob * advantage)
        value_losses.append(F.smooth_l1_loss(value, torch.tensor([R], device=device)))

    optimizer.zero_grad()
    loss = torch.stack(policy_losses).sum() + torch.stack(value_losses).sum()

    loss.backward()
    optimizer.step()

    del model.rewards[:]
    del model.saved_actions[:]
    model.integral = 0
    model.prev_error = 0
    model.u_prev = 0

In [5]:
def select_action(state):
    error = state[0] - state[1]
    model.integral += error
    model.derivative = error - model.prev_error

    state = torch.tensor([state[0], state[1], state[2], state[3], state[4], model.u_prev, error, model.prev_error, model.integral, model.derivative], device=device, dtype=torch.float32)
    probs, state_value = model(state)
    m = Categorical(probs)
    action = m.sample()
    model.saved_actions.append(SavedAction(m.log_prob(action), state_value))

    model.prev_error = error
    model.u_prev = action.item()
    return action.item()


In [6]:
action_space_n = 30
actions = np.linspace(STEER_RANGE[0], STEER_RANGE[1], action_space_n + 1)

model = ActorCritic(action_space_n=action_space_n, observation_space_n=10).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-2)
schedular = optim.lr_scheduler.StepLR(optimizer, 20, 0.5)
eps = np.finfo(np.float32).eps.item()

In [7]:
def train():
    env = Environment()
    for i_episode in range(300):
        state = env.reset()
        ep_reward = 0
        for step in count():
            action = select_action(state)
            state, reward, terminated = env.step(action)
            model.rewards.append(reward)
            ep_reward += reward

            if terminated:
                break

        finish_episode()
        print(f"Episode {i_episode} Reward: {ep_reward:.2f}")

In [8]:
pid = PIDController()
env = Environment()
state = env.reset()
ep_reward = 0
for step in count():
    action = pid.update(state[0], state[1], [state[2], state[3], state[4]])
    state, reward, terminated = env.step(action)
    model.rewards.append(reward)
    ep_reward += reward

    if terminated:
        break

print(ep_reward)

2024-05-19 05:02:24.914135304 [E:onnxruntime:Default, provider_bridge_ort.cc:1480 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1193 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.11: cannot open shared object file: No such file or directory

2024-05-19 05:02:24.914170304 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:747 CreateExecutionProviderInstance] Failed to create CUDAExecutionProvider. Please reference https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements to ensure all dependencies are met.


-17.282048723796382


In [9]:
train()

2024-05-19 04:32:11.265620900 [E:onnxruntime:Default, provider_bridge_ort.cc:1480 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1193 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.11: cannot open shared object file: No such file or directory

2024-05-19 04:32:11.265659300 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:747 CreateExecutionProviderInstance] Failed to create CUDAExecutionProvider. Please reference https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements to ensure all dependencies are met.


Episode 0 Reward: -10209.90
Episode 1 Reward: -10502.84
Episode 2 Reward: -10502.84
Episode 3 Reward: -10502.84
Episode 4 Reward: -10502.84
Episode 5 Reward: -10502.84
Episode 6 Reward: -10505.59
Episode 7 Reward: -10502.84
Episode 8 Reward: -10487.13
Episode 9 Reward: -10476.78
Episode 10 Reward: -10482.67
Episode 11 Reward: -10483.21
Episode 12 Reward: -10485.17
Episode 13 Reward: -10457.56
Episode 14 Reward: -10484.79
Episode 15 Reward: -10324.70
Episode 16 Reward: -10452.46
Episode 17 Reward: -10502.84
Episode 18 Reward: -10502.84
Episode 19 Reward: -10502.84
Episode 20 Reward: -10502.84
Episode 21 Reward: -10497.27
Episode 22 Reward: -10502.84
Episode 23 Reward: -10496.32
Episode 24 Reward: -10502.84
Episode 25 Reward: -10502.41
Episode 26 Reward: -10502.84
Episode 27 Reward: -10492.44
Episode 28 Reward: -10501.26
Episode 29 Reward: -10490.07
Episode 30 Reward: -10505.07
Episode 31 Reward: -10496.91
Episode 32 Reward: -10502.84
Episode 33 Reward: -10502.84
Episode 34 Reward: -1050

In [10]:
torch.save(model, './models/actor_critic.pth')